In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: Eine Qiskit Function von Q-CTRL Fire Opal
*Siehe die [API-Referenz](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Qiskit Functions sind ein experimentelles Feature, das ausschließlich Nutzerinnen und Nutzern des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über IBM Quantum Platform API) Plan zur Verfügung steht. Sie befinden sich im Preview-Release-Status und können sich noch ändern.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>

## Überblick
Mit dem Fire Opal Optimization Solver kannst du Optimierungsprobleme im Utility-Scale auf Quantenhardware lösen, ohne Quantenexpertise zu benötigen. Gib einfach die übergeordnete Problemdefinition ein, und der Solver übernimmt den Rest. Der gesamte Workflow ist rauschbewusst und nutzt intern [Fire Opals Performance Management](/guides/q-ctrl-performance-management). Der Solver liefert zuverlässig genaue Lösungen für klassisch anspruchsvolle Probleme, auch im vollen Geräteumfang auf den größten IBM&reg; QPUs.

Der Solver ist flexibel und kann eingesetzt werden, um kombinatorische Optimierungsprobleme zu lösen, die als Zielfunktionen oder beliebige Graphen definiert sind. Probleme müssen nicht auf die Gerätetopologie abgebildet werden. Sowohl unbeschränkte als auch beschränkte Probleme sind lösbar, sofern Einschränkungen als Strafterme formuliert werden können. Die in diesem Leitfaden enthaltenen Beispiele zeigen, wie man ein unbeschränktes und ein beschränktes Optimierungsproblem im Utility-Scale mit verschiedenen Solver-Eingabetypen löst. Das erste Beispiel umfasst ein max-cut-Problem, das auf einem 156-Knoten-3-regulären Graphen definiert ist, während das zweite Beispiel ein 50-Knoten-Minimum-Vertex-Cover-Problem angeht, das durch eine Kostenfunktion definiert ist.

Um Zugang zum Optimization Solver zu erhalten, [kontaktiere Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Beschreibung der Function
Der Solver optimiert und automatisiert den gesamten Algorithmus vollständig – von der Fehlerunterdrückung auf Hardwareebene bis hin zu effizientem Problem-Mapping und geschlossener klassischer Optimierung. Im Hintergrund reduziert die Pipeline des Solvers Fehler auf jeder Stufe und ermöglicht so die verbesserte Leistung, die für eine bedeutungsvolle Skalierung erforderlich ist. Der zugrundeliegende Workflow ist vom Quantum Approximate Optimization Algorithm (QAOA) inspiriert, einem hybriden quanten-klassischen Algorithmus. Eine detaillierte Zusammenfassung des vollständigen Optimization-Solver-Workflows findest du im [veröffentlichten Manuskript](https://arxiv.org/abs/2406.01743).

![Visualisierung des Optimization-Solver-Workflows](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

So löst du ein allgemeines Problem mit dem Optimization Solver:
1. Definiere dein Problem als Zielfunktion, einen Graphen oder eine `SparsePauliOp`-Spinkette.
2. Verbinde dich mit der Function über den Qiskit Functions Catalog.
3. Führe das Problem mit dem Solver aus und rufe die Ergebnisse ab.
### Akzeptierte Problemformate
- Polynomielle Ausdrucksdarstellung einer Zielfunktion. Idealerweise in Python mit einem vorhandenen SymPy-Poly-Objekt erstellt und mit [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr) in einen String formatiert.
- Graphdarstellung eines bestimmten Problemtyps. Der Graph sollte mit der networkx-Bibliothek in Python erstellt werden. Er wird dann durch Verwendung der networkx-Funktion `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)` in einen String umgewandelt.
- Spinkettendarstellung eines bestimmten Problems. Die Spinkette sollte als `SparsePauliOp`-Objekt dargestellt werden; weitere Details findest du in der [Dokumentation](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp).

> **Note:** Wenn du ein Backend verwenden möchtest, das diese Funktion derzeit nicht unterstützt, [wende dich an Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com), um Support hinzuzufügen.
## Benchmarks
[Veröffentlichte Benchmarking-Ergebnisse](https://arxiv.org/abs/2406.01743) zeigen, dass der Solver Probleme mit über 120 Qubits erfolgreich löst und dabei sogar zuvor veröffentlichte Ergebnisse auf Quanten-Annealing- und Trapped-Ion-Geräten übertrifft. Die folgenden Benchmark-Metriken geben einen groben Hinweis auf die Genauigkeit und Skalierung von Problemtypen anhand einiger Beispiele. Die tatsächlichen Metriken können je nach verschiedenen Problemeigenschaften variieren, wie z. B. der Anzahl der Terme in der Zielfunktion (Dichte) und deren Lokalität, der Anzahl der Variablen und der polynomiellen Ordnung.

Die angegebene „Anzahl der Qubits" ist keine absolute Begrenzung, sondern stellt ungefähre Schwellenwerte dar, bei denen du eine äußerst konsistente Lösungsgenauigkeit erwarten kannst. Größere Problemgrößen wurden erfolgreich gelöst, und Tests jenseits dieser Grenzen sind ausdrücklich erwünscht.

Beliebige Qubit-Konnektivität wird für alle Problemtypen unterstützt.

| Problemtyp    | Anzahl der Qubits | Beispiel | Genauigkeit | Gesamtzeit (s) | Runtime-Nutzung (s) | Anzahl der Iterationen
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Dünn verbundene quadratische Probleme  | 156 | 3-reguläres max-cut | 100%     | 1764     | 293          | 16 |
| Optimierung binärer Probleme höherer Ordnung | 156 | Ising-Spinglas-Modell | 100%      | 1461     | 272          | 16 |
| Dicht verbundene quadratische Probleme | 50 | Vollständig verbundenes max-cut | 100%      |  1758    | 268  | 12 |
| Beschränktes Problem mit Straftermen | 50 | Gewichtetes Minimum Vertex Cover mit 8 % Kantendichte | 100%      | 1074     | 215 | 10 |
## Erste Schritte
Authentifiziere dich zunächst mit deinem [IBM Quantum API-Schlüssel](http://quantum.cloud.ibm.com/). Wähle dann die Qiskit Function wie folgt aus. (Dieser Codeausschnitt setzt voraus, dass du dein Konto bereits [in deiner lokalen Umgebung gespeichert hast](/guides/functions#install-qiskit-functions-catalog-client).)

In [4]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Das Problem definieren
Du kannst ein max-cut-Problem ausführen, indem du ein Graphenproblem definierst und `problem_type='maxcut'` angibst.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [1]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [2]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. Das Problem ausführen
Wenn du die graphenbasierte Eingabemethode verwendest, gib den Problemtyp an.

In [3]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [4]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [9]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

34b53970-d95a-4e24-8763-fc6f3d112843


QUEUED


### 3. Das Ergebnis abrufen
Rufe den optimalen Schnittwert aus dem Ergebnis-Dictionary ab.

> **Note:** Die Zuordnung der Variablen zum Bitstring kann sich geändert haben. Das Ausgabe-Dictionary enthält ein `variables_to_bitstring_index_map`-Unter-Dictionary, das hilft, die Reihenfolge zu überprüfen.

In [10]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can also solve constrained optimization problems by passing hard constraints directly to the Solver through the `constraint` input, instead of encoding them as penalty terms in the objective function. The Solver currently supports Hamming-weight-1 constraints: each constraint specifies a group of variables where exactly one variable must equal 1 and the rest must equal 0.

The following example demonstrates how to construct a cost function and a set of hard constraints for a constrained optimization problem, [graph partitioning](https://en.wikipedia.org/wiki/Graph_partition), by assigning every node in a graph to exactly one of several groups while minimizing the total weight of edges whose endpoints land in the same group.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [11]:
# %pip install numpy networkx sympy

Du kannst die Genauigkeit des Ergebnisses überprüfen, indem du das Problem klassisch mit Open-Source-Solvern wie [PuLP](https://coin-or.github.io/pulp/) löst, sofern der Graph nicht dicht verbunden ist. Bei Problemen mit hoher Dichte sind möglicherweise fortgeschrittene klassische Solver erforderlich, um die Lösung zu validieren.
## Beispiel: Beschränkte Optimierung
Das vorherige max-cut-Beispiel ist ein gängiges quadratisches unbeschränktes binäres Optimierungsproblem. Q-CTRLs Optimization Solver kann für verschiedene Problemtypen eingesetzt werden, einschließlich beschränkter Optimierung. Du kannst beliebige Problemtypen lösen, indem du die Problemdefinition als Polynom eingibst, bei dem Einschränkungen als Strafterme modelliert werden.

Das folgende Beispiel zeigt, wie eine Kostenfunktion für ein beschränktes Optimierungsproblem, das [Minimum Vertex Cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC), konstruiert wird.
Zusätzlich zu den Paketen `qiskit-ibm-catalog` und `qiskit` verwendest du in diesem Beispiel auch die folgenden Pakete: `numpy`, `networkx` und `sympy`. Du kannst diese Pakete installieren, indem du die folgende Zelle auskommentierst, wenn du dieses Beispiel in einem Notebook mit dem IPython-Kernel ausführst.

In [26]:
import networkx as nx
from sympy import Symbol, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
min_weight = -1.0
max_weight = 1.0
for i in graph.nodes:
    weight = (max_weight - min_weight) * _rng.random() + min_weight
    graph.add_node(i, weight=weight)

# Optionally, visualize the graph
nx.draw_networkx(graph, nx.kamada_kawai_layout(graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Das Problem definieren
Definiere ein zufälliges MVC-Problem, indem du einen Graphen mit zufällig gewichteten Knoten erzeugst.

In [27]:
# Construct the cost function.
group_count = 3
variables = [
    Symbol(f"n[{i},{g}]")
    for i in range(node_count)
    for g in range(group_count)
]
node_group_var = {
    (i, g): variables[i * group_count + g]
    for i in range(node_count)
    for g in range(group_count)
}
cost_function = Poly(0, *variables)

for i, j in graph.edges():
    edge_weight = graph.nodes[i]["weight"] + graph.nodes[j]["weight"]
    for g in range(group_count):
        cost_function += (
            edge_weight * node_group_var[(i, g)] * node_group_var[(j, g)]
        )

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Ein Standardoptimierungsmodell für gewichtetes MVC kann wie folgt formuliert werden. Zunächst muss eine Strafe für jeden Fall hinzugefügt werden, in dem eine Kante nicht mit einem Knoten in der Teilmenge verbunden ist. Sei daher $n_i = 1$, wenn Knoten $i$ zur Überdeckung gehört (d. h. in der Teilmenge ist), und $n_i = 0$ andernfalls. Das Ziel ist es, die Gesamtzahl der Knoten in der Teilmenge zu minimieren, was durch die folgende Funktion dargestellt werden kann:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [28]:
# Build the hard constraint: exactly one group per node.
constraint_dict = {
    str(tuple(f"n[{i},{g}]" for g in range(group_count))): 1
    for i in range(node_count)
}
print(f"Problem constraints: {constraint_dict}")

Problem constraints: {"('n[0,0]', 'n[0,1]', 'n[0,2]')": 1, "('n[1,0]', 'n[1,1]', 'n[1,2]')": 1, "('n[2,0]', 'n[2,1]', 'n[2,2]')": 1, "('n[3,0]', 'n[3,1]', 'n[3,2]')": 1, "('n[4,0]', 'n[4,1]', 'n[4,2]')": 1, "('n[5,0]', 'n[5,1]', 'n[5,2]')": 1, "('n[6,0]', 'n[6,1]', 'n[6,2]')": 1, "('n[7,0]', 'n[7,1]', 'n[7,2]')": 1, "('n[8,0]', 'n[8,1]', 'n[8,2]')": 1, "('n[9,0]', 'n[9,1]', 'n[9,2]')": 1, "('n[10,0]', 'n[10,1]', 'n[10,2]')": 1, "('n[11,0]', 'n[11,1]', 'n[11,2]')": 1, "('n[12,0]', 'n[12,1]', 'n[12,2]')": 1, "('n[13,0]', 'n[13,1]', 'n[13,2]')": 1, "('n[14,0]', 'n[14,1]', 'n[14,2]')": 1, "('n[15,0]', 'n[15,1]', 'n[15,2]')": 1, "('n[16,0]', 'n[16,1]', 'n[16,2]')": 1, "('n[17,0]', 'n[17,1]', 'n[17,2]')": 1, "('n[18,0]', 'n[18,1]', 'n[18,2]')": 1, "('n[19,0]', 'n[19,1]', 'n[19,2]')": 1, "('n[20,0]', 'n[20,1]', 'n[20,2]')": 1, "('n[21,0]', 'n[21,1]', 'n[21,2]')": 1, "('n[22,0]', 'n[22,1]', 'n[22,2]')": 1, "('n[23,0]', 'n[23,1]', 'n[23,2]')": 1, "('n[24,0]', 'n[24,1]', 'n[24,2]')": 1, "('n[25,

Jetzt sollte jede Kante im Graphen mindestens einen Endpunkt der Überdeckung enthalten, was als Ungleichung ausgedrückt werden kann:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Jeder Fall, in dem eine Kante nicht mit dem Knoten der Überdeckung verbunden ist, muss bestraft werden. Dies kann in der Kostenfunktion durch Hinzufügen einer Strafe der Form $P(1-n_i-n_j+n_i n_j)$ dargestellt werden, wobei $P$ eine positive Strafkonstante ist. Damit ist eine unbeschränkte Alternative zur beschränkten Ungleichung für gewichtetes MVC:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [20]:
# Solve the problem
partition_job = solver.run(
    problem=srepr(cost_function),
    constraint=constraint_dict,
    backend_name="ibm_marrakesh",  # E.g. "ibm_marrakesh"
)

### 2. Das Problem ausführen

In [21]:
# Print the ID so you can use it later, if necessary
print(partition_job.job_id)

# Get job status
print(partition_job.status())

b8085944-f313-444e-be39-ea61b1b47ebd
QUEUED


Überprüfe den [Status](/guides/functions#check-job-status) deiner Qiskit-Function-Arbeitslast oder rufe [Ergebnisse](/guides/functions#retrieve-results) wie folgt ab:

In [ ]:
partition_result = partition_job.result()
qctrl_cost = partition_result["solution_bitstring_cost"]
solution_bitstring = partition_result["solution_bitstring"]

# Print results
print(f"Total weight of same-group edges: {qctrl_cost}")
print(f"Solution bitstring: {solution_bitstring}")

Total weight of same-group edges: -36.5539
Solution bitstring: 100100100100100001100100100100100100100100100100100001010100010100100100100010001001100100100001100001100001010001001010100100100100100010100100100100


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-08-10: Added support for hard (Hamming weight 1) constraints via the `constraint` input, and updated the constrained optimization example to use them.
- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>